<a href="https://colab.research.google.com/github/SaurabhAmbhore/repo1/blob/main/Ankercloud_olap3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark findspark
import findspark
findspark.init()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.2-py2.py3-none-any.whl size=317812365 sha256=958de953dc56032a709114d7631b969363a38af1cea4f4667042730f336e44fb
  Stored in directory: /root/.cache/pip/wheels/34/34/bd/03944534c44b677cd5859f248090daa9fb27b3c8f8e5f49574
Successfully built pyspark


In [ ]:
# Necessary Imports
from pyspark.sql import SparkSession
from google.cloud import storage
from datetime import datetime, timedelta
import re
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, LongType, DoubleType, TimestampType, \
    DateType, FloatType, IntegerType

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Bookings and Payments OLAP") \
    .config("spark.dynamicAllocation.enabled", True) \
    .config("spark.shuffle.service.enabled", True) \
    .getOrCreate()


In [ ]:
df_product_regions = spark.read.parquet("/content/postgres_raw_dump_olap3_product_regions_part-00009-0f8c9b0a-ce7d-40c2-9d59-3bcc03f16f17-c000.snappy.parquet")
df_products = spark.read.parquet("/content/postgres_raw_dump_olap3_products_part-00001-e07126c3-8cc3-473e-8f6b-3beb9f3ae97c-c000.snappy.parquet")

In [ ]:
# print("Schema of df_product_regions")
# df_product_regions.printSchema()
# print("Schema of df_products")
# df_products.printSchema()

In [ ]:
product_regions_df = df_product_regions.select("region_id","name","product_id").groupBy("name") \
                                .agg(count("product_id").alias("total_number_of_products")) \
                                .distinct().drop(df_product_regions.id)

product_regions_df.printSchema()
df_product_regions.printSchema()

root
 |-- name: string (nullable = true)
 |-- total_number_of_products: long (nullable = false)

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- region_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [ ]:
product_regions_df = product_regions_df.join(df_product_regions, on="name", how="left").drop(df_product_regions.id, df_product_regions.region_id)

product_regions_df.printSchema()

root
 |-- name: string (nullable = true)
 |-- total_number_of_products: long (nullable = false)
 |-- id: long (nullable = true)
 |-- region_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [ ]:
# df_products.printSchema()
# product_regions_df.printSchema()

# Drop unnecessary columns from products_df
products_df = df_products.join(product_regions_df, product_regions_df.product_id == df_products.id, how="left") \
     .drop("product_id").drop(product_regions_df.id)

# Print the schema to verify the selected columns
products_df.printSchema()
print(products_df.count())


root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
# Calculate the count of active products per id
products_df = products_df.withColumn(
    "number_of_active_products",
    count(when(col("state") == "active", 1)).over(Window.partitionBy("id"))
)

# Calculate the count of inactive products per id
products_df = products_df.withColumn(
    "number_of_inactive_products",
    count(when(col("state") == "inactive", 1)).over(Window.partitionBy("id"))
)
# Calculate the number_of_template_Itineraries per id
products_df = products_df.withColumn(
    "number_of_template_Itineraries",
    count(when(col("is_customized") == "false", 1)).over(Window.partitionBy("id"))
)

# Print the schema to verify the columns
products_df.printSchema()

products_df.show(5, truncate=False)

root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
df_active_product_count = df_active_products.groupBy("id").agg(count("id").alias("number_of_active_products"))
df_inactive_product_count = df_inactive_products.groupBy("id").agg(count("id").alias("number_of_inactive_products"))
# df_active_product_count.printSchema()
# df_inactive_product_count.show(5)

In [ ]:
df_enquiries = spark.read.parquet("/content/postgres_raw_dump_olap3_enquiries_part-00000-2028f577-9825-4d9e-9fec-3f80b758c904-c000.snappy.parquet")
df_enquiries.printSchema()

root
 |-- id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- destination_id: long (nullable = true)
 |-- old_system_product_slug: string (nullable = true)
 |-- region_name: string (nullable = true)
 |-- old_system_region_id: long (nullable = true)
 |-- date_of_travel: date (nullable = true)
 |-- number_of_pax: integer (nullable = true)
 |-- old_system_dump: string (nullable = true)
 |-- origin_source: string (nullable = true)
 |-- page_source: string (nullable = true)
 |-- section_source: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- migrated_from_old_system: boolean (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- referrer: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- budget_type: string (nullable = tru

In [ ]:

products_and_enquiries_df = products_df.join(df_enquiries,
                                             products_df.id == df_enquiries.product_id,
                                             how="left") \
                                       .drop(df_enquiries.id)

# # Print the schema to verify the columns
# products_and_enquiries_df.printSchema()

# # Show the DataFrame to verify the results
# products_and_enquiries_df.show(5)

In [ ]:

# Define the window specification for partitioning by 'region_id'
window_spec = Window.partitionBy("region_id")

# Add column for total number of pax and average number of pax by region
products_and_enquiries_df = products_and_enquiries_df \
    .withColumn("total_number_of_pax", count(col("number_of_pax")).over(window_spec)) \
    .withColumn("average_pax", avg(col("number_of_pax")).over(window_spec)) \
    .withColumn("median_pax_by_region", median(col("number_of_pax")).over(window_spec))

# Print the schema to verify the result
products_and_enquiries_df.printSchema()
products_and_enquiries_df.show(5)

root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
df_booking_line_items = spark.read.parquet("/content/postgres_raw_dump_olap3_booking_line_items_part-00011-353b3aa4-e8d3-4a71-baaa-605e6f2dfb4b-c000.snappy.parquet")
df_booking_line_items.printSchema

root
 |-- id: long (nullable = true)
 |-- booking_id: long (nullable = true)
 |-- resource_type: string (nullable = true)
 |-- resource_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: float (nullable = true)
 |-- unit_price: float (nullable = true)
 |-- no_of_adults: integer (nullable = true)
 |-- no_of_children: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- strike_through_amount: float (nullable = true)
 |-- code: string (nullable = true)
 |-- currency: string (nullable = true)



In [ ]:
# Join `products_and_enquiries_df` with `df_booking_line_items` on the `id` column
product_enquiries_bookingli_df = products_and_enquiries_df \
    .join(df_booking_line_items, products_and_enquiries_df.product_id == df_booking_line_items.id, how="left") \
    .drop(df_booking_line_items.id)
# Print the schema to verify the result
product_enquiries_bookingli_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
# Define the window specification for partitioning by 'id'
window_spec = Window.partitionBy("id")

# Add the column `no_of_bookings_solo_travellers` with the sum of quantities for solo travellers
product_enquiries_bookingli_df = product_enquiries_bookingli_df.withColumn(
    "no_of_bookings_solo_travellers",
    sum(when((col("resource_type") == "Thrillo::Common::BookedInventory") & (col("quantity") == 1), col("quantity")).otherwise(lit(0))).over(window_spec)
)

# Print the schema to verify the new column
product_enquiries_bookingli_df.printSchema()

# Show the first few rows to check the result
# product_enquiries_bookingli_df.show(5)

root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
# Add the column `no_of_bookings_couple_travellers` with the sum of quantities for couple travellers
product_enquiries_bookingli_df = product_enquiries_bookingli_df.withColumn(
    "no_of_bookings_couple_travellers",
    sum(
        when(
            (col("resource_type") == "Thrillo::Common::BookedInventory") &
            (col("quantity") == 2) &
            (col("number_of_adults") == 2),
            col("quantity")
        ).otherwise(lit(0))
    ).over(window_spec)
)

# Print the schema to verify the new column
product_enquiries_bookingli_df.printSchema()

# Show the first few rows to check the result
product_enquiries_bookingli_df.show(5)

root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
product_enquiries_bookingli_df = product_enquiries_bookingli_df.withColumn(
    "no_of_bookings_small_family_travellers",
    sum(
        when(
            (col("resource_type") == "Thrillo::Common::BookedInventory") &
            (col("quantity") == 3) &
            (col("number_of_adults") == 2)&
            (col("number_of_kids")==1),
            col("quantity")
        ).otherwise(lit(0))
    ).over(window_spec)
)
# Print the schema to verify the new column
product_enquiries_bookingli_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
product_enquiries_bookingli_df = product_enquiries_bookingli_df.withColumn(
    "no_of_bookings_big_family_travellers",
    sum(
        when(
            (col("resource_type") == "Thrillo::Common::BookedInventory") &
            (col("quantity") == 4) &
            (col("number_of_adults") == 2)&
            (col("number_of_kids")==2),
            col("quantity")
        ).otherwise(lit(0))
    ).over(window_spec)
)
# Print the schema to verify the new column
product_enquiries_bookingli_df.printSchema()
product_enquiries_bookingli_df.show(5,truncate=False)

root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
product_enquiries_bookingli_df = product_enquiries_bookingli_df.withColumn(
    "no_of_bookings_group_travellers",
    sum(
        when(
            (col("resource_type") == "Thrillo::Common::BookedInventory") &
            (col("quantity") <= 30) &
            (col("number_of_adults") > 2),
            col("quantity")
        ).otherwise(lit(0))
    ).over(window_spec)
)
# Print the schema to verify the new column
product_enquiries_bookingli_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- old_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- booking_confirmation_type: string (nullable = true)
 |-- min_percentage_amount_to_confirm: integer (nullable = true)
 |-- starting_price: float (nullable = true)
 |-- strike_through_price: float (nullable = true)
 |-- partner_id: long (nullable = true)
 |-- code: string (nullable = true)
 |-- steps_completed: string (nullable = true)
 |-- cancellation_policy_id: long (nullable = true)
 |-- payment_term_policy_id: long (nullable = true)
 |-- confirmation_policy_id: long (nullable = true)
 |-- refund_policy_id: long (nullable = true)
 |-- enable_online_booking: boolean (nullable = true)
 |-- e

In [ ]:
product_enquiries_bookingli_df.show(5,truncate=False)

+---+-------------------------------------------------+---------------------------------------------------------+-------------------------+--------+--------------------------+--------------------------+--------+--------+------------+-------------------------+--------------------------------+--------------+--------------------+----------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+----------------------+----------------------+----------------+---------------------+-------------------+--------------+-------------------+----------------------+------------------+-----------------------+-------------+-----------------+------------------------+-----------------+----------------------------------+-------------+----------